# Evaluating a DSpark Draft Model for Qwen3-0.6B

This notebook measures the **DSpark drafter** trained in the sibling notebooks
([`yosefw/Qwen3-0.6B-DSpark`](https://huggingface.co/yosefw/Qwen3-0.6B-DSpark)) when it is used for
speculative decoding with [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B) as the
verifier.

The drafter guesses a block of tokens ahead, the verifier checks the whole block in one forward
pass and keeps what it agrees with - same output, fewer verifier passes. The number that decides
whether that is worth doing is the **acceptance length**: how many tokens survive per verification
round. For the full explanation of speculative decoding and DSpark, see
[`[online] train-dspark-drafter-qwen-3-0.6b.ipynb`](%5Bonline%5D%20train-dspark-drafter-qwen-3-0.6b.ipynb).

**Hardware.** Tested on Kaggle with **2x T4**; the server below is started data-parallel across
both.

**Pipeline**

1. **Setup** - clone `speculators` and install it alongside vLLM, plus the evaluation extras.
2. **Serve** - bring up the drafter in vLLM, which pulls in its verifier automatically.
3. **Evaluate** - run `evaluate.py throughput` against that endpoint to get acceptance metrics.
4. **Shut down** - free the GPUs.

Based on the official
[evaluating performance tutorial](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/evaluating_performance/).

### Setup your environment

The evaluation harness ships inside the same
[speculators](https://github.com/vllm-project/speculators) repository as the training scripts, so
the setup here mirrors the training notebooks: clone the repo, `cd` into it, install it together
with vLLM.

**Every relative path used later** (`./scripts`, `./output`) is relative to this repository root -
on Kaggle that is `/kaggle/working/speculators`.

In [ ]:
!git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

Create the virtual environment and install `speculators` in editable mode alongside
`vllm>=0.22.0`.

> **Note:** `source speculators_venv/bin/activate` inside a `!` cell only affects that one
> subshell - it does **not** persist to later cells. Everything below therefore runs against the
> notebook's own interpreter, which is why the packages are installed there too.

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

Two Kaggle-specific fix-ups:

- **`pip uninstall torchaudio`** - the pre-installed build is pinned to Kaggle's `torch` and breaks
  the import chain once vLLM upgrades it. Nothing here needs audio.
- **`pip install -Uq datasets`** - the evaluation datasets need a newer `datasets` than the image
  ships. `--break-system-packages` is required because the image's site-packages is managed.

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

The evaluation harness lives in `scripts/evaluate` and has its own `requirements.txt` on top of the
main install (the `guidellm` benchmark client and dataset loaders). `cd` into it - `evaluate.py` is
run from this directory in the cells below.

In [ ]:
%cd scripts/evaluate
! pip install -r requirements.txt

### Serve the model under test

Start vLLM on the **drafter** repo. A checkpoint in speculators format carries its verifier in its
config, so vLLM loads `Qwen3-0.6B` as the target and wires up the full speculative-decoding stack
from this one model id - no separate `--speculative-config` needed.

Swap in the commented-out `./output/checkpoints/checkpoint_best` to evaluate a checkpoint you just
trained locally instead of the published copy.

| flag | meaning |
| --- | --- |
| `--port 8000` | where the OpenAI-compatible API listens |
| `--gpu-memory-utilization 0.75` | leaves headroom on each T4 for the drafter and CUDA graphs |
| `--data-parallel-size 2` | one replica per T4, so the benchmark can keep both busy |

`start_new_session=True` puts the server in its own process group, so interrupting a notebook cell
will **not** kill it; logs go to `/content/vllm_serve.log`. The loop then polls `/health` up to 120
times, 5 s apart (~10 minutes) - the first start has to download weights and capture CUDA graphs.
If it never reports ready, read the log (`!tail -n 50 /content/vllm_serve.log`); out-of-memory
errors and port conflicts both show up there.

In [ ]:
import subprocess, os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_file = open("/content/vllm_serve.log", "w")
process = subprocess.Popen(
    [
        "vllm", "serve", "yosefw/Qwen3-0.6B-DSpark", #"./output/checkpoints/checkpoint_best",
        "--port", "8000",
        "--gpu-memory-utilization", "0.75",
        "--data-parallel-size", "2",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,  # detaches from notebook's process group so cell interrupts don't kill it
)
print(f"Started vLLM server with PID {process.pid}")

import time, requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

### Run the evaluation

`evaluate.py` has two subcommands, and this notebook uses the cheaper one:

| subcommand | what it does |
| --- | --- |
| **`throughput`** (used here) | reports **acceptance metrics only**, skipping the benchmark sweep - despite the name, this is the quick option |
| `sweep` | the full pipeline: output-length estimation plus a performance sweep across all 9 subsets of [`RedHatAI/speculator_benchmarks`](https://huggingface.co/datasets/RedHatAI/speculator_benchmarks), written to `perf_results_<timestamp>/perf_results.csv` |

`--target` is the only required flag; everything else defaults, including the dataset (all 9
subsets), `--max-concurrency 128`, and `--max-requests 200`. Useful overrides: `--subsets` to
evaluate a single category, `--gen-kwargs '{"temperature":0.6}'` to change decoding, and
`--output-dir` to pin the results directory.

Results land in `acceptance.csv`: per-subset **`acceptance_length`** (mean tokens kept per
verification round) and **`acceptance_at_pos_N`** (how often the Nth drafted token in a block
survives). Acceptance length is capped at 5 here because the drafter was trained with
`--block-size 4`, and per-position acceptance decays across the block - slowing that decay is
exactly what DSpark's semi-autoregressive head is for. This drafter's numbers are tabulated in the
repository [`README.md`](../README.md).

> To turn these into a speedup figure, run `sweep` here and against the same server started
> without the drafter, then compare with
> `python plot.py speedup --baseline "No Spec=nospec/perf_results.csv" --target "DSpark=dspark/perf_results.csv" --metric latency`.

In [ ]:
! python evaluate.py throughput --target http://localhost:8000/v1

Shut the server down. `os.killpg` targets the whole process group created by
`start_new_session=True`, so the vLLM workers go with it - a plain `process.kill()` would leave
children holding GPU memory.

Run this once evaluation finishes (or before re-running the server cell) to free the GPUs and port
8000.

In [ ]:
import signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)